# 10 — Audit the frozen V3 ensemble

This stage loads the already-frozen five-fold models and thresholds, then
evaluates the original seven test subjects once. Because those subjects were
inspected during V2 development, OOF performance remains the primary estimate;
this result is an audit comparison rather than a pristine final test.

In [ ]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
RAW_DIR = Path(os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")).expanduser().resolve()
V3_DATA_DIR = Path(os.environ.get("PURE_V3_DATA_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-v3")).expanduser().resolve()
V3_MODEL_DIR = Path(os.environ.get("PURE_V3_MODEL_DIR", REPO_ROOT / "models" / "production" / "v3")).expanduser().resolve()

print("Raw:", RAW_DIR)
print("V3 data:", V3_DATA_DIR)
print("V3 models:", V3_MODEL_DIR)

In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "evaluate_v3_ensemble.py"),
        "--input-dir", str(V3_DATA_DIR),
        "--model-dir", str(V3_MODEL_DIR),
    ],
    check=True,
)

In [ ]:
import json
import pandas as pd

report = json.loads((V3_MODEL_DIR / "test_metrics.json").read_text())
display(pd.Series(report["test_binary"]).to_frame("audit test"))
display(pd.Series(report["test_gate"]).to_frame("audit gate"))
display(pd.DataFrame(report["branch_test_metrics"]).T[["accuracy", "balanced_accuracy", "f1_good", "roc_auc", "false_acceptance_rate"]])
print("Reliable precision:", report["reliable_precision"])